# Beijing station data ETL

This notebook reads the 12 PRSA station tables from the MySQL `air_quality` database, combines their records, checks station coverage and missingness, and prepares a cleaned dataset. It exports the cleaned data to `cleaned_dataset.csv` and replaces the MySQL `all_regions` table with the combined cleaned records.

Run the cells in order. The notebook expects the station tables to exist in MySQL and the PRSA source CSV files to be available in one of the searched `data` folders. Review the cleaning rules and output checks before using the resulting data downstream.

## 1. Import libraries

Import pandas for tabular processing and SQLAlchemy for database connections.

In [1]:
import os
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine

## 2. Connect to the database

Create a connection to the local MySQL database `air_quality` and check whether a connection can be opened. Later cells read from and write to this database.

In [2]:
engine = create_engine(
    "mysql+pymysql://root:@localhost/air_quality"
)

try:
    with engine.connect():
        print("Connected to air_quality!")
except Exception as error:
    print(f"Failed to connect to air_quality: {error}")

Connected to air_quality!


## 3. Locate the PRSA station CSV files

Search the candidate `data` folders for files matching `PRSA_Data_*.csv` and list the files found. The notebook stops with an error if it cannot find any matching files.

In [3]:
working_dir = Path.cwd().resolve()
candidate_folders = [
    working_dir / "data",
    working_dir.parent / "data",
    working_dir / "air-quality-classification" / "data",
]
dataset_folder = next(
    (
        folder
        for folder in candidate_folders
        if folder.is_dir() and any(folder.glob("PRSA_Data_*.csv"))
    ),
    None,
)
if dataset_folder is None:
    checked_folders = ", ".join(str(folder) for folder in candidate_folders)
    raise FileNotFoundError(
        f"Could not find PRSA station CSV files. Checked: {checked_folders}"
    )

files = sorted(dataset_folder.glob("PRSA_Data_*.csv"))

print(f"Datasets found: {len(files)} in {dataset_folder}")

for file in files:
    print(file.name)
candidate_folders = [
    working_dir / "data",
    working_dir.parent / "data",
    working_dir / "air-quality-classification" / "data",
]
dataset_folder = next(
    (
        folder
        for folder in candidate_folders
        if folder.is_dir() and any(folder.glob("PRSA_Data_*.csv"))
    ),
    None,
)
if dataset_folder is None:
    checked_folders = ", ".join(str(folder) for folder in candidate_folders)
    raise FileNotFoundError(
        f"Could not find PRSA station CSV files. Checked: {checked_folders}"
    )

files = sorted(dataset_folder.glob("PRSA_Data_*.csv"))

print(f"Datasets found: {len(files)} in {dataset_folder}")

for file in files:
    print(os.path.basename(file))

Datasets found: 12 in C:\Users\PLP\Desktop\DataScienceProjects\Data_Science\air-quality-classification\data
PRSA_Data_Aotizhongxin_20130301-20170228.csv
PRSA_Data_Changping_20130301-20170228.csv
PRSA_Data_Dingling_20130301-20170228.csv
PRSA_Data_Dongsi_20130301-20170228.csv
PRSA_Data_Guanyuan_20130301-20170228.csv
PRSA_Data_Gucheng_20130301-20170228.csv
PRSA_Data_Huairou_20130301-20170228.csv
PRSA_Data_Nongzhanguan_20130301-20170228.csv
PRSA_Data_Shunyi_20130301-20170228.csv
PRSA_Data_Tiantan_20130301-20170228.csv
PRSA_Data_Wanliu_20130301-20170228.csv
PRSA_Data_Wanshouxigong_20130301-20170228.csv
Datasets found: 12 in C:\Users\PLP\Desktop\DataScienceProjects\Data_Science\air-quality-classification\data
PRSA_Data_Aotizhongxin_20130301-20170228.csv
PRSA_Data_Changping_20130301-20170228.csv
PRSA_Data_Dingling_20130301-20170228.csv
PRSA_Data_Dongsi_20130301-20170228.csv
PRSA_Data_Guanyuan_20130301-20170228.csv
PRSA_Data_Gucheng_20130301-20170228.csv
PRSA_Data_Huairou_20130301-20170228.csv

## 4. Preview a station dataset

Inspect the dimensions and first rows of a station file to confirm its structure before loading or combining station data.

In [4]:
## Preview a Station Dataset


## 5. Load station data into MySQL

Load each station CSV into its corresponding MySQL table. The tables are then used as the sources for the combined dataset.

In [5]:
## Load Each Station into MySQL


In [6]:
query = "SELECT COUNT(*) AS total_records FROM aotizhongxin"

result = pd.read_sql(query, engine)

result

,total_records
0,35064


## 6. Combine the 12 station tables

Read the expected station tables from MySQL, record each table's row count, and concatenate the rows into one DataFrame. Keep a raw copy for the cleaning step and compare the combined row count with the sum of the source counts.

In [ ]:
table_names = [
    "aotizhongxin", "changping", "dingling", "dongsi",
    "guanyuan", "gucheng", "huairou", "nongzhanguan",
    "shunyi", "tiantan", "wanliu", "wanshouxigong"
]

dataframes = []
source_row_counts = {}

for table_name in table_names:
    dataframe = pd.read_sql_query(
        f"SELECT * FROM `{table_name}`",
        engine
    )
    dataframes.append(dataframe)
    source_row_counts[table_name] = len(dataframe)
    print(f"{table_name}: {len(dataframe):,} rows")

df_combined = pd.concat(dataframes, ignore_index=True)

df_combined_raw = df_combined.copy(deep=True)


source_total_rows = sum(source_row_counts.values())
combined_total_rows = len(df_combined)

print(f"\nRows from all source tables: {source_total_rows:,}")
print(f"Rows in df_combined: {combined_total_rows:,}")
print(f"Row counts match: {source_total_rows == combined_total_rows}")

aotizhongxin: 35,064 rows
changping: 35,064 rows
dingling: 35,064 rows
dongsi: 35,064 rows
guanyuan: 35,064 rows
gucheng: 35,064 rows
huairou: 35,064 rows


## 7. Check station coverage

Compare the station names in the combined data with the expected list of 12 PRSA stations. The check reports missing or unexpected names; it does not modify the data.

In [ ]:
expected_stations = {
    "aotizhongxin", "changping", "dingling", "dongsi",
    "guanyuan", "gucheng", "huairou", "nongzhanguan",
    "shunyi", "tiantan", "wanliu", "wanshouxigong"
}

if "station" not in df_combined.columns:
    raise KeyError("The 'station' column was not found in df_combined.")

present_stations = set(df_combined["station"].dropna().str.lower().unique())
missing_stations = expected_stations - present_stations
unexpected_stations = present_stations - expected_stations

print(f"Expected stations: {len(expected_stations)}")
print(f"Stations found: {len(present_stations)}")
print(f"Missing stations: {sorted(missing_stations) or 'None'}")
print(f"Unexpected stations: {sorted(unexpected_stations) or 'None'}")

if not missing_stations and not unexpected_stations:
    print("All expected stations are present.")
else:
    print("The station list does not exactly match the expected stations.")

Expected stations: 12
Stations found: 12
Missing stations: None
Unexpected stations: None
All expected stations are present.


## 8. Inspect duplicates and missing values

Count exact duplicate rows and report null values in identification and measurement columns. Also inspect missing hours, rain values, and rows with more than half of the measurement fields missing. This is a diagnostic step; duplicate rows are not removed by this check.

In [ ]:
required_id_cols = ["No", "year", "month", "day", "station"]
measurement_cols = [
    "PM2.5", "PM10", "SO2", "NO2", "CO", "O3",
    "TEMP", "PRES", "DEWP", "RAIN", "wd", "WSPM"
]

if not set(required_id_cols).issubset(df_combined.columns):
    missing_required = set(required_id_cols) - set(df_combined.columns)
    raise KeyError(f"Missing required columns: {sorted(missing_required)}")

duplicate_rows = df_combined.duplicated().sum()
print(f"Duplicate rows before cleaning: {duplicate_rows}")

required_nulls = {
    col: int(df_combined[col].isna().sum())
    for col in required_id_cols
}

print("\nNull counts in required identification columns:")
for col, count in required_nulls.items():
    print(f"  {col}: {count}")

all_null_counts = df_combined.isna().sum()
null_columns = all_null_counts[all_null_counts > 0]

print("\nNull counts in all columns with null values:")
if null_columns.empty:
    print("  No null values found in any column.")
else:
    for col, count in null_columns.items():
        print(f"  {col}: {count}")

hour_null_count = int(df_combined["hour"].isna().sum()) if "hour" in df_combined.columns else 0
print(f"\nNull hour count: {hour_null_count}")

if "hour" in df_combined.columns:
    zero_hour_groups = set(
        df_combined.loc[df_combined["hour"] == 0, ["year", "month", "day", "station"]]
        .drop_duplicates()
        .itertuples(index=False, name=None)
    )

    null_hour_duplicates = (
        df_combined.loc[df_combined["hour"].isna()]
        .apply(
            lambda row: (
                row["year"], row["month"], row["day"], row["station"]
            ) in zero_hour_groups,
            axis=1
        )
        .sum()
    )

    print(f"Null-hour rows that duplicate an existing 0-hour entry: {null_hour_duplicates}")

rain_null_count = int(df_combined["RAIN"].isna().sum()) if "RAIN" in df_combined.columns else 0
print(f"Null RAIN count before fill: {rain_null_count}")

if measurement_cols:
    missing_ratio = df_combined[measurement_cols].isna().mean(axis=1)
    rows_above_threshold = int((missing_ratio > 0.5).sum())
    print(f"Rows with >50% missing measurement values: {rows_above_threshold}")
else:
    print("No measurement columns available for missingness threshold check.")

Duplicate rows before cleaning: 0

Null counts in required identification columns:
  No: 0
  year: 0
  month: 0
  day: 0
  station: 0

Null counts in all columns with null values:
  PM2.5: 8739
  PM10: 6449
  SO2: 9021
  NO2: 12116
  CO: 20701
  O3: 13277
  TEMP: 398
  PRES: 393
  DEWP: 403
  RAIN: 390
  wd: 1822
  WSPM: 318

Null hour count: 0
Null-hour rows that duplicate an existing 0-hour entry: 0.0
Null RAIN count before fill: 390
Rows with >50% missing measurement values: 26


## 9. Clean missing measurements

Create a cleaned copy of the combined data and record which fields are filled. Rows with more than half of the listed measurements missing are dropped. Pollutant and selected weather gaps are linearly interpolated within each station for gaps of up to three rows; rain nulls are filled with zero only when at least 75% of its non-null values are zero; wind direction uses forward/back filling for up to three rows. Rows still missing all six pollutant values are dropped. Other remaining nulls are reported, not universally removed. The cleaned result is exported as `cleaned_dataset.csv`.

In [ ]:
df_cleaned = df_combined_raw.copy(deep=True)
df_cleaned["interpolated_columns"] = [[] for _ in range(len(df_cleaned))]

required_id_cols = ["No", "year", "month", "day", "station"]
measurement_cols = [
    "PM2.5", "PM10", "SO2", "NO2", "CO", "O3",
    "TEMP", "PRES", "DEWP", "RAIN", "wd", "WSPM"
]
pollutant_cols = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]
meteorological_cols = ["TEMP", "PRES", "DEWP", "WSPM"]

if not set(required_id_cols).issubset(df_cleaned.columns):
    missing_required = set(required_id_cols) - set(df_cleaned.columns)
    raise KeyError(f"Missing required columns: {sorted(missing_required)}")

row_count_before_step1 = len(df_cleaned)
missing_ratio = df_cleaned[measurement_cols].isna().mean(axis=1)
df_cleaned = df_cleaned.loc[missing_ratio <= 0.5].copy()
row_count_after_step1 = len(df_cleaned)
rows_dropped_by_missing_ratio = int((missing_ratio > 0.5).sum())
print(f"Rows before >50% missing removal: {row_count_before_step1:,}")
print(f"Rows after >50% missing removal: {row_count_after_step1:,}")
print(f"Rows dropped by >50% missing measurement rule: {rows_dropped_by_missing_ratio:,}")

df_cleaned = df_cleaned.sort_values(
    ["station", "year", "month", "day", "hour"],
    kind="mergesort"
    ).reset_index(drop=True)

for col in pollutant_cols:
    if col not in df_cleaned.columns:
        continue
    missing_before = df_cleaned[col].isna().copy()
    df_cleaned[col] = (
        df_cleaned.groupby("station", group_keys=False)[col]
        .transform(
            lambda s: s.interpolate(method="linear", limit=3, limit_direction="both")
        )
    )
    interpolated_mask = missing_before & df_cleaned[col].notna()
    df_cleaned.loc[interpolated_mask, "interpolated_columns"] = (
        df_cleaned.loc[interpolated_mask, "interpolated_columns"]
        .apply(lambda names: names + [col])
    )

for col in meteorological_cols:
    if col not in df_cleaned.columns:
        continue
    missing_before = df_cleaned[col].isna().copy()
    df_cleaned[col] = (
        df_cleaned.groupby("station", group_keys=False)[col]
        .transform(
            lambda s: s.interpolate(method="linear", limit=3, limit_direction="both")
        )
    )
    interpolated_mask = missing_before & df_cleaned[col].notna()
    df_cleaned.loc[interpolated_mask, "interpolated_columns"] = (
        df_cleaned.loc[interpolated_mask, "interpolated_columns"]
        .apply(lambda names: names + [col])
    )

if "RAIN" in df_cleaned.columns:
    rain_non_null = df_cleaned["RAIN"].dropna()
    rain_zero_share = (rain_non_null == 0).mean() if len(rain_non_null) > 0 else 0.0
    print(f"RAIN zero share among non-null values: {rain_zero_share:.2%}")

    if rain_zero_share >= 0.75:
        rain_missing_before = df_cleaned["RAIN"].isna().copy()
        df_cleaned.loc[rain_missing_before, "RAIN"] = 0
        print("RAIN nulls filled with 0 because the series is predominantly dry/zero.")
    else:
        print("RAIN is not predominantly zero; nulls were left for review instead of interpolating.")

if "wd" in df_cleaned.columns:
    wd_missing_before = df_cleaned["wd"].isna().copy()
    df_cleaned["wd"] = (
        df_cleaned.groupby("station", group_keys=False)["wd"]
        .transform(lambda s: s.ffill(limit=3).bfill(limit=3))
    )
    interpolated_mask = wd_missing_before & df_cleaned["wd"].notna()
    df_cleaned.loc[interpolated_mask, "interpolated_columns"] = (
        df_cleaned.loc[interpolated_mask, "interpolated_columns"]
        .apply(lambda names: names + ["wd"])
    )
    remaining_wd_nulls = int(df_cleaned["wd"].isna().sum())

all_pollutant_missing_after = df_cleaned[pollutant_cols].isna().all(axis=1)
rows_dropped_by_all_pollutants_missing = int(all_pollutant_missing_after.sum())

df_cleaned = df_cleaned.loc[~all_pollutant_missing_after].copy()
print(f"Rows dropped because all six pollutant columns are still missing after interpolation: {rows_dropped_by_all_pollutants_missing:,}")

remaining_null_counts = df_cleaned.isna().sum().sort_values(ascending=False)
remaining_null_counts = remaining_null_counts[remaining_null_counts > 0]

if remaining_null_counts.empty:
    print("  No nulls remain in the cleaned dataset.")
else:
    print("\nRemaining null counts per column after interpolation/fill:")
    print(remaining_null_counts)

rows_with_nulls = int(df_cleaned.isna().any(axis=1).sum())
print(f"\nRows still containing nulls after all steps: {rows_with_nulls:,}")

nulls_per_row = df_cleaned[measurement_cols].isna().sum(axis=1)
for count in range(nulls_per_row.max() + 1):
    rows = (nulls_per_row == count).sum()
    print(f"Rows with exactly {count} null(s): {rows:,}")

six_null_rows = df_cleaned[nulls_per_row == 6].copy()
if not six_null_rows.empty:
    missing_pattern = (
        six_null_rows[measurement_cols]
        .isna()
        .astype(int)
        .astype(str)
        .agg("".join, axis=1)
    )
    print("\nMissing-value patterns among rows with exactly 6 nulls after cleaning:")
    print(missing_pattern.value_counts())
else:
    print("\nNo rows with exactly 6 missing measurement values remain after cleaning.")

df_cleaned["interpolated_columns"] = df_cleaned["interpolated_columns"].apply(
    lambda names: ", ".join(names) if names else "None"
 )

df_combined = df_cleaned.copy(deep=True)

display(df_combined[["station", "year", "month", "day", "hour", "interpolated_columns"]].head())

df_combined.to_csv("cleaned_dataset.csv", index=False, encoding="utf-8-sig")

print("Cleaned dataset successfully exported to cleaned_dataset.csv")


Rows before >50% missing removal: 420,768
Rows after >50% missing removal: 420,742
Rows dropped by >50% missing measurement rule: 26
RAIN zero share among non-null values: 96.07%
RAIN nulls filled with 0 because the series is predominantly dry/zero.
Rows dropped because all six pollutant columns are still missing after interpolation: 1,989

Remaining null counts per column after interpolation/fill:
CO       9996
NO2      3533
O3       3436
SO2      1901
PM2.5    1359
PM10      299
wd          8
DEWP        1
PRES        1
TEMP        1
dtype: int64

Rows still containing nulls after all steps: 17,212
Rows with exactly 0 null(s): 401,541
Rows with exactly 1 null(s): 15,387
Rows with exactly 2 null(s): 1,098
Rows with exactly 3 null(s): 88
Rows with exactly 4 null(s): 507
Rows with exactly 5 null(s): 132

No rows with exactly 6 missing measurement values remain after cleaning.


,station,year,month,day,hour,interpolated_columns
0,Aotizhongxin,2013,3,1,0,None
1,Aotizhongxin,2013,3,1,1,None
2,Aotizhongxin,2013,3,1,2,None
3,Aotizhongxin,2013,3,1,3,None
4,Aotizhongxin,2013,3,1,4,None


Cleaned dataset successfully exported to cleaned_dataset.csv


## 10. Verify the cleaned dataset

Review the cleaned row and column counts and display the resulting DataFrame. The cleaning cell above also prints remaining null counts and records which fields were interpolated or filled.

In [ ]:
verification_display = df_combined.copy()

print(f"Before Cleaning Rows: {len(df_combined_raw):,}")
print(f"Final DataFrame rows: {len(df_combined):,}")
print(f"Final DataFrame columns: {len(df_combined.columns)}")
print("\nFinal DataFrame Verification Table:")
display(verification_display)


Before Cleaning Rows: 420,768
Final DataFrame rows: 418,753
Final DataFrame columns: 19

Final DataFrame Verification Table:


,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station,interpolated_columns
0,1,2013,3,1,0,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,Aotizhongxin,None
1,2,2013,3,1,1,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7,Aotizhongxin,None
2,3,2013,3,1,2,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6,Aotizhongxin,None
3,4,2013,3,1,3,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1,Aotizhongxin,None
4,5,2013,3,1,4,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0,Aotizhongxin,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
420737,35060,2017,2,28,19,11.0,32.0,3.0,24.0,400.0,72.0,12.5,1013.5,-16.2,0.0,NW,2.4,Wanshouxigong,None
420738,35061,2017,2,28,20,13.0,32.0,3.0,41.0,500.0,50.0,11.6,1013.6,-15.1,0.0,WNW,0.9,Wanshouxigong,None
420739,35062,2017,2,28,21,14.0,28.0,4.0,38.0,500.0,54.0,10.8,1014.2,-13.3,0.0,NW,1.1,Wanshouxigong,None
420740,35063,2017,2,28,22,12.0,23.0,4.0,30.0,400.0,59.0,10.5,1014.4,-12.9,0.0,NNW,1.2,Wanshouxigong,None


## 11. Write the combined data to MySQL

Replace the MySQL `all_regions` table with the cleaned combined DataFrame, then compare the table's row count with the DataFrame row count. This operation overwrites the existing `all_regions` table.

In [ ]:
df_combined.to_sql(
    "all_regions",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000
)

loaded_count = pd.read_sql_query(
    "SELECT COUNT(*) AS total_rows FROM all_regions",
    engine
)

print(f"Rows in final DataFrame: {len(df_combined):,}")
print(f"Rows in MySQL all_regions table: {loaded_count['total_rows'].iloc[0]:,}")
print(
    f"Load row counts match: "
    f"{loaded_count['total_rows'].iloc[0] == len(df_combined)}"
)


Rows in final DataFrame: 418,753
Rows in MySQL all_regions table: 418,753
Load row counts match: True


## 12. Query station-level average PM2.5

Calculate the average PM2.5 value for each station in `all_regions` and display the five highest averages. The top result describes the records currently stored in the table; it can be affected by the date range, missingness, and cleaning choices. It is a descriptive comparison, not a causal explanation of station differences.

In [ ]:
query = """
SELECT
    station,
    AVG(`PM2.5`) AS average_pm25
FROM all_regions
GROUP BY station
ORDER BY average_pm25 DESC;
"""

result = pd.read_sql_query(query, engine)

top_5 = result.head(5)
display(top_5)

top_station = result.iloc[0]

print(
    f"\nTop 1 Station: {top_station['station']}"
    f" with an average PM2.5 concentration of "
    f"{top_station['average_pm25']:.2f}"
)

,station,average_pm25
0,Dongsi,86.113104
1,Nongzhanguan,85.115528
2,Wanshouxigong,84.976929
3,Gucheng,83.922580
4,Wanliu,83.428047



Top 1 Station: Dongsi with an average PM2.5 concentration of 86.11
